# Insight rule validation

This notebook contains small, runnable examples that exercise the prototype rule functions in `src/rules`.
Use these cells to validate thresholds, inspect supporting stats, and generate example insight JSON for integration.

In [ ]:
# Make local package importable when running from the notebooks folder
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

# Imports: rule functions and evaluator
from src.metrics.baseline import MetricResult
from src.rules.index import energy_uplift_after_strength, late_meals_lower_morning_energy, routine_stabilizes_mood
from src.rules.evaluators import evaluate_rules_for_user

print('Imports OK')

In [ ]:
# Example: energy uplift after strength — construct short and long MetricResult objects
short = MetricResult(metric='post_energy_short', window_days=7, value=5.2, data_points=5)
long = MetricResult(metric='post_energy_long', window_days=30, value=3.8, data_points=10)

res = energy_uplift_after_strength(short, long, threshold=0.8, min_points=3)
print('Energy uplift rule result:')
print(res)


In [ ]:
# Example: late meals correlate with lower morning energy (prototype)
from datetime import datetime, time
# synthetic meal times (assume datetime objects) and next-morning energies
meal_times = [datetime(2026, 2, 20, 22, 15), datetime(2026, 2, 22, 20, 30), datetime(2026, 2, 24, 23, 5)]
morning_energies = [3, 4, 2, 3, 4, 3]  # sample morning scores 1-5

res = late_meals_lower_morning_energy(meal_times, morning_energies, window_days=14, threshold=-0.5, min_points=3)
print('Late meals rule result:')
print(res)


In [ ]:
# Example: routine stabilizes mood — compare variance on routine vs non-routine days
# routine_stress = stress measurements on days with workout+meal; nonroutine_stress otherwise
routine_stress = [2.1, 2.3, 1.9, 2.0, 2.2]
nonroutine_stress = [3.2, 2.9, 3.5, 3.0, 2.8]

res = routine_stabilizes_mood(routine_stress, nonroutine_stress, min_points=3, reduction_ratio=0.9)
print('Routine stabilizes mood result:')
print(res)


In [ ]:
# Use the high-level evaluator to run available rules for a synthetic user
metrics = {'post_energy_short': short, 'post_energy_long': long}
extra = {'meal_times': meal_times, 'morning_energies': morning_energies}
insights = evaluate_rules_for_user('user_123', metrics=metrics, extra_inputs=extra)
print('Evaluator returned insights:')
for i in insights:
    print(i)


Next steps:
- Replace the placeholder alignment logic for `late_meals_lower_morning_energy` with real alignment of meal->next-morning energy.
- Add sensitivity cells that sweep thresholds and plot precision/recall on synthetic labels.
- Export example insight JSON using `run/run_evaluator.py` (see README).